## Cel i Architektura Notatnika

Niniejszy notatnik stanowi bezpośrednią kontynuację i rozwinięcie wstępnych prac analitycznych zrealizowanych na 10-procentowej próbie badawczej.

Zgodnie z przyjętą strategią projektową oraz zaleceniami promotora, w tym etapie wprowadzono następujące zmiany architektoniczne i metodologiczne:

- Rezygnacja z próby i praca na 100% populacji: Cały potok przetwarzania danych został przepustowo przeskalowany na pełny, oryginalny zbiór danych US Accidents (liczący blisko 7,8 miliona rekordów), co pozwala na pełne wykorzystanie potencjału uogólniania algorytmów.

- Modularność kodu (Refaktoryzacja): Logika wczytywania, oczyszczania i przekształcania danych została wydzielona do zewnętrznego modułu inżynieryjnego (data_pipeline.py). Dzięki temu notatnik pozostaje czytelny, a poszczególne funkcje są wielokrotnie używalne bez duplikacji kodu.

- Zarządzanie zasobami: W potoku przetwarzania zastosowano jawne czyszczenie pamięci operacyjnej (gc.collect()) oraz mechanizmy wektoryzacyjne, co ułatwi pracę z wielkimi macierzami cech.

- ...

In [2]:
# Importujemy własną bibliotekę!
from data_pipeline import przygotuj_dane

# Odpalamy 100% danych
X_train, X_test, y_train, y_test = przygotuj_dane("C:/Users/Goral/OneDrive/Pulpit/praca inż/US_Accidents_March23.csv")

1. Wczytywanie danych z pliku: C:/Users/Goral/OneDrive/Pulpit/praca inż/US_Accidents_March23.csv
   Początkowy rozmiar danych: 7728394 wierszy.
2. Standaryzacja nazw kolumn
3. Tworzenie zmiennej docelowej (Severity_Binary)
4. Ekstrakcja cech czasowych
5. Korekta i uzupełnianie braków danych
6. Usuwanie zbędnych kolumn i szumu informacyjnego...
   Rozmiar danych po czyszczeniu: 6816415 wierszy.
7. Kodowanie zmiennych logicznych i tekstowych
8. Podział na zbiór treningowy i testowy (80/20)
Zakończono! Wymiary X_train: (5453132, 39), X_test: (1363283, 39)


## Strategia postępowania z niezbalansowanym zbiorem danych

Jak wykazano w fazie eksploracyjnej (EDA) w 1 pliku, rozkład zmiennej docelowej Severity_Binary charakteryzuje się silną asymetrią – klasa 0 (wypadki niegroźne) dominuje, podczas gdy klasa 1 (wypadki groźne) stanowi mniejszość. Standardowe algorytmy uczenia maszynowego dążą do maksymalizacji ogólnej dokładności (Accuracy), co w przypadku tak niezbalansowanych populacji prowadzi do patologicznego ignorowania klasy mniejszościowej.


### 1. Metody na poziomie funkcji straty (Cost-Sensitive Learning / Wagowanie klas)

Zamiast manipulować strukturą samych danych (co mogłoby zniekształcić naturalne wzorce), zmodyfikowano wewnętrzną funkcję straty (Loss Function) algorytmów.


W klasycznej klasyfikacji binarnej zminimalizowaniu podlega standardowa binarna entropia krzyżowa (Log-Loss):

$$L(y, p) = -\frac{1}{N} \sum_{i=1}^{N} \left[ y_i \log(p_i) + (1 - y_i) \log(1 - p_i) \right]$$

W warunkach silnego niezbalansowania, błędy popełniane na klasie mniejszościowej giną w ogólnej sumie kosztów. Aby temu zapobiec, stosuje się podejście Cost-Sensitive Learning, wprowadzając do funkcji straty wagę penalizującą(/karzącą) błędne predykcje dla klasy rzadkiej:

$$L_{weighted}(y, p) = -\frac{1}{N} \sum_{i=1}^{N} \left[ w_1 \cdot y_i \log(p_i) + w_0 \cdot (1 - y_i) \log(1 - p_i) \right]$$

Gdzie waga $w_1$ dla klasy mniejszościowej jest wyliczana proporcjonalnie do odwrotności liczności klas:

$$w_1 = \frac{N}{2 \cdot N_1}$$

Dzięki temu błąd popełniony przy klasyfikacji rzadkiej klasy 1 jest dla modelu surowiej karany, co wymusza na algorytmie poszukiwanie cech charakterystycznych dla najgroźniejszych zdarzeń. W projekcie parametr ten aktywowano m.in. poprzez `class_weight='balanced'` w bibliotece scikit-learn.

### Eksperyment Badawczy: Wpływ narzędzi radzenia sobie z niezbalansowaniem klas

W celu empirycznego udowodnienia zasadności zastosowania metod kompensujących asymetrię danych na pełnym zbiorze treningowym (100%), przeprowadzono eksperyment porównawczy. Przetestowano ten sam algorytm (Drzewo Decyzyjne) w dwóch konfiguracjach:

1. `Model Baseline (Bez korekcji):` Trenowany na surowym rozkładzie danych, dążący do maksymalizacji ogólnej dokładności (Accuracy).

2. `Model Korygowany (Z wagowaniem klas):` Trenowany z użyciem parametru `class_weight='balanced'`, modyfikującym funkcję straty.

In [3]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report

print("Eksperyment 1: Model bez narzędzi radzenia sobie z niezbalansowaniem")
# Trenujemy standardowe drzewo bez class_weight na 100% danych
dt_niezbalansowany = DecisionTreeClassifier(max_depth=10, random_state=42)
dt_niezbalansowany.fit(X_train, y_train)
preds_niezbalansowany = dt_niezbalansowany.predict(X_test)
print(classification_report(y_test, preds_niezbalansowany))

print("\nEksperyment  2: Model z narzędziami (Wagowanie klas)")
# Trenujemy drzewo z włączonym balansem klas na 100% danych
dt_zbalansowany = DecisionTreeClassifier(class_weight='balanced', max_depth=10, random_state=42)
dt_zbalansowany.fit(X_train, y_train)
preds_zbalansowany = dt_zbalansowany.predict(X_test)
print(classification_report(y_test, preds_zbalansowany))

Eksperyment 1: Model bez narzędzi radzenia sobie z niezbalansowaniem
              precision    recall  f1-score   support

           0       0.87      0.91      0.89   1072740
           1       0.61      0.48      0.54    290543

    accuracy                           0.82   1363283
   macro avg       0.74      0.70      0.71   1363283
weighted avg       0.81      0.82      0.82   1363283


Eksperyment  2: Model z narzędziami (Wagowanie klas)
              precision    recall  f1-score   support

           0       0.94      0.71      0.81   1072740
           1       0.44      0.85      0.58    290543

    accuracy                           0.74   1363283
   macro avg       0.69      0.78      0.70   1363283
weighted avg       0.84      0.74      0.76   1363283



### Wnioski analityczne z eksperymentu na pełnym zbiorze danych:
1. Iluzoryczna skuteczność modelu niekorygowanego: W wariancie pierwszym (bez korekcji wag klas) model osiąga wysoką ogólną dokładność (Accuracy na poziomie 82%), co w powierzchownej analizie mogłoby sugerować poprawność predykcji. Jednak szczegółowa weryfikacja wskaźników dla klasy mniejszościowej (wypadki groźne, klasa 1) ujawnia jej drastyczne niedoszacowanie – czułość (Recall) wynosi zaledwie 0.48. Oznacza to, że algorytm optymalizujący wyłącznie ogólną dokładność systematycznie ignoruje ponad połowę najpoważniejszych incydentów drogowych.

2. Skuteczność mechanizmu kosztowego (Cost-Sensitive Learning): Wprowadzenie do funkcji straty wag karzących za błędy na klasie mniejszościowej (Eksperyment 2) doprowadziło do fundamentalnej zmiany charakterystyki predykcyjnej modelu. Wskaźnik czułości (Recall) dla klasy groźnych wypadków wzrósł skokowo z 0.48 do 0.85.

3. Kompromis metodologiczny (Precision-Recall Trade-off): Zastosowanie wagowania klas wiąże się ze spadkiem ogólnego Accuracy (do 74%) oraz obniżeniem precyzji dla klasy 1 (z 0.61 do 0.44). Jest to jednak w pełni kontrolowany i pożądany kompromis inżynieryjny.

## Przejście na Klasyfikację Wieloklasową (Multiclass)

W początkowej fazie badawczej, zgodnie z podejściem iteracyjnym i w celu stabilnego przetestowania struktur danych oraz optymalizacji przy pełnym zbiorze, problem został uproszczony do klasyfikacji binarnej (wypadki niegroźne: klasy 1-2 vs. groźne: klasy 3-4). Pozwoliło to na weryfikację mechanizmów radzenia sobie z asymetrią klas.

Jednakże oryginalny zbiór `US Accidents` charakteryzuje się 4-stopniową, porządkową skalą ciężkości zdarzeń (Severity od 1 do 4). Aby praca posiadała pełną wartość naukową, w kolejnym kroku badawczym rezygnujemy z binaryzacji i wracamy do pełnej klasyfikacji wieloklasowej, co pozwala na uchwycenie subtelniejszych różnic między stopniami zagrożenia.

#### Przejście od Sigmoidu do Funkcji Softmax

W klasyfikacji binarnej optymalizowaliśmy prawdopodobieństwo przynależności do jednej klasy przy użyciu funkcji aktywacji Sigmoid. W przypadku klasyfikacji wieloklasowej, model musi wygenerować wektor prawdopodobieństw dla każdej z $C$ kategorii (gdzie $C = 4$), sumujący się do 1.

1. Kombinacja liniowa (Logity):
Dla każdej klasy $k \in \{1, 2, \dots, C\}$ model wylicza wynik liniowy (logit) $z_k$:

$$z_k = w_k^T x + b_k$$

Gdzie $x$ to wektor cech wejściowych danej obserwacji, $w_k$ to wektor wag przypisanych do klasy $k$, a $b_k$ to wyraz wolny (bias) dla klasy $k$.

2. Funkcja Softmax:
Aby przekształcić surowe wyniki liniowe (logity) w legalny rozkład prawdopodobieństw (gdzie wartości mieszczą się w przedziale $[0, 1]$ i sumują się do 1), stosuje się funkcję Softmax:

$$P(y = k \mid x) = \sigma(z)_k = \frac{e^{z_k}}{\sum_{j=1}^{C} e^{z_j}}$$

Gdzie $e^{z_k}$ to eksponenta logitu danej klasy $k$, a mianownik stanowi sumę eksponent logitów wszystkich $C$ dostępnych klas, co zapewnia normalizację do rozkładu prawdopodobieństwa.

3. Wieloklasowa Entropia Krzyżowa (Multi-Class Cross-Entropy Loss):
Funkcja straty (kryterium optymalizacyjne) dla całego zbioru $N$ obserwacji przyjmuje postać uogólnioną na $C$ klas:

$$L = -\frac{1}{N} \sum_{i=1}^{N} \sum_{k=1}^{C} y_{i,k} \log(P(y_i = k \mid x_i))$$

Gdzie:

$N$ – całkowita liczba obserwacji w zbiorze danych,

$C$ – całkowita liczba klas (w naszym przypadku $C = 4$),

$y_{i,k}$ – zmienna wskaźnikowa (one-hot encoded), przyjmująca wartość $1$, jeśli $i$-ta obserwacja faktycznie należy do klasy $k$, oraz $0$ w przeciwnym razie,

$P(y_i = k \mid x_i)$ – wyliczone przez model Softmax prawdopodobieństwo, że $i$-ta obserwacja należy do klasy $k$.

Co zmienia się technicznie w potoku przetwarzania?

1. Zmienna docelowa: Zamiast kolumny binarnej, modelem objaśnianym staje się oryginalna, 4-stopniowa kolumna Severity. Dla kompatybilności z wieloma bibliotekami (np. Scikit-Learn, XGBoost), etykiety mapuje się z zakresu $(1, 2, 3, 4)$ na $(0, 1, 2, 3)$.

2. Ewaluacja: Macierz błędów zmienia wymiar z $2 \times 2$ na $4 \times 4$, co pozwala nam precyzyjnie zbadać, które poziomy ciężkości (np. mylenie klasy 3 z klasą 4) generują największe błędy predykcyjne.


In [5]:
# Importujemy naszą nową bibliotekę dla 4 klas Severity
from data_pipeline_multiclass import przygotuj_dane_multiclass

# Uruchomienie preprocessingu na 100% danych dla Multiclass
X_train, X_test, y_train, y_test = przygotuj_dane_multiclass("US_Accidents_March23.csv")

1. Wczytywanie danych z pliku: US_Accidents_March23.csv
Początkowy rozmiar danych: 7728394 wierszy.
2. Standaryzacja nazw kolumn.
3. Utrzymanie oryginalnej zmiennej docelowej (Severity: 1, 2, 3, 4).
4. Ekstrakcja cech czasowych.
5. Korekta i uzupełnianie braków danych.
6. Usuwanie zbędnych kolumn i szumu informacyjnego.
   Rozmiar danych po czyszczeniu: 6816415 wierszy.
7. Kodowanie zmiennych logicznych i tekstowych.
8. Podział na zbiór treningowy i testowy (80/20) z zachowaniem stratyfikacji.
Zakończono. Wymiary X_train: (5453132, 39), X_test: (1363283, 39) 


## Uwaga metodyczna: Wpływ skalowania zmiennych numerycznych na modele liniowe i drzewiaste a interpretowalność

W procesie projektowania potoku przetwarzania danych (Data Preprocessing Pipeline) wdrożono zróżnicowane podejście do skalowania zmiennych numerycznych, zależne od specyfiki wykorzystywanej grupy algorytmów:

- Modele liniowe i metryczne (np. Regresja Logistyczna): W przypadku modeli opartych na spadku gradientu i obliczaniu dystansu geometrycznego, wstępna standaryzacja numeryczna (np. StandardScaler / transformacja Z-Score) jest wymogiem matematycznym. Zapobiega ona wydłużaniu kształtu funkcji straty (Log-Loss) i pozwala na prawidłową oraz stabilną zbieżność algorytmu optymalizacyjnego.

- Algorytmy oparte na zespołach drzew (Gradient Boosting): Dla modeli docelowych (XGBoost, LightGBM, CatBoost) celowo zrezygnowano ze skalowania. Ponieważ drzewa decyzyjne operują na progach podziału i są całkowicie niezmiennicze wobec monotonicznych transformacji, normalizacja wartości nie wpływa na jakość predykcji. Co więcej, pozostawienie cech w ich naturalnych, fizycznych jednostkach (np. stopnie Fahrenheita, prędkość w mph, godziny doby) ma fundamentalne znaczenie dla interpretowalności modeli. Umożliwia to bezstratną ekstrakcję wiedzy (analizę ważności cech – Feature Importance), bezpośrednio wiążąc wyniki matematyczne z realnymi uwarunkowaniami ruchu drogowego.